In [63]:
import geojson
from shapely import Polygon
import os

In [64]:
files = {}
usa_counties = "/home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/usa/counties.geojson"
# files[usa_counties] = "Counties"
usa_congressional = "/home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/usa/congressional.geojson"
files[usa_congressional] = "Congressional"
usa_outline = "/home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/usa/outline.geojson"
files[usa_outline] = "Outline"
usa_states = "/home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/usa/states.geojson"
files[usa_states] = "States"

france_counties = "/home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/france/counties.geojson"
files[france_counties] = "France Counties"
france_states = "/home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/france/states.geojson"
files[france_states] = "France States"

germany_counties = "/home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/germany/counties.geojson"
files[germany_counties] = "Germany Counties"
germany_states = "/home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/germany/states.geojson"
files[germany_states] = "Germany States"    

netherlands_counties = "/home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/netherlands/counties.geojson"
files[netherlands_counties] = "Netherlands Counties"
netherlands_states = "/home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/netherlands/states.geojson"
files[netherlands_states] = "Netherlands States"

world_countries = "/home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/world/world.geojson"
files[world_countries] = "World Countries"
world = "/home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/world/custom.geojson"
files[world] = "World"

In [65]:
for file in files:
    with open(file) as f:
        gj = geojson.load(f)
        # print(type(gj["features"]))
        # for feature in gj["features"]:
            # print(feature)
        # print(gj["features"][0]["properties"])
        print(f"KEYS of {files[file]}: {gj["features"][0]["geometry"]["coordinates"]}"[:100])
        print(len(gj["features"][0]["geometry"]["coordinates"]))
        print(len(gj["features"][0]["geometry"]["coordinates"][0]))
       # print(gj["features"][0]["properties"]["NAME"])

KEYS of Congressional: [[[-93.097296, 40.584014], [-93.098507, 40.583973], [-93.135802, 40.582854], 
1
376
KEYS of Outline: [[-122.75802, 49.002357], [-122.407829, 49.002193], [-122.405989, 49.002239], [-122
34
2
KEYS of States: [[[[-67.619761, 44.519754], [-67.61541, 44.521973], [-67.587738, 44.516196], [-67.58
42
1
KEYS of France Counties: [[[[4.131665, 49.974987], [4.121902, 49.973846], [4.117551, 49.972253], [4.
2
1
KEYS of France States: [[[4.215945, 49.954411], [4.218054, 49.938041], [4.216999, 49.916794], [4.224
1
523
KEYS of Germany Counties: [[[[8.364335, 54.867987], [8.362575, 54.868327], [8.361542, 54.868749], [8
15
1
KEYS of Germany States: [[[12.101344, 50.31398], [12.100998, 50.314371], [12.093712, 50.322613], [12
1
745
KEYS of Netherlands Counties: [[[[6.648593, 53.026333], [6.644658, 53.039387], [6.640518, 53.041725]
1
1
KEYS of Netherlands States: [[[6.41328, 52.985523], [6.362521, 53.033969], [6.367811, 53.06736], [6.
1
223
KEYS of World Countries: [[[[117.703608, 4.1

In [66]:
# with open(usa_states) as f:
#     gj = geojson.load(f)
#     print(type(gj["features"]))
#     # for feature in gj["features"]:
#         # print(feature)
#     print(gj["features"][0]["properties"])
#     print(gj["features"][0]["properties"].keys())
#     print(gj["features"][0]["properties"]["NAME"])

In [67]:
def normalize_geometry(feature):
    geom = feature.get("geometry", {})
    coords = geom.get("coordinates")

    if not coords:
        return feature

    # Helper: detect nesting depth
    def get_depth(x):
        depth = 0
        while isinstance(x, list):
            if len(x) == 0:
                break
            x = x[0]
            depth += 1
        return depth

    depth = get_depth(coords)

    # --- Handle different geometry types safely ---

    # Case 1: Point → ignore
    if depth == 1:
        return feature

    # Case 2: LineString → wrap into Polygon (optional)
    if depth == 2:
        coords = [coords]  # make it a ring
        geom["type"] = "Polygon"

    # Case 3: Polygon → already fine
    elif depth == 3:
        geom["type"] = "Polygon"

    # Case 4: MultiPolygon → take first polygon
    elif depth >= 4:
        coords = coords[0]  # ⚠️ drops extra polygons
        geom["type"] = "Polygon"

    geom["coordinates"] = coords
    feature["geometry"] = geom

    return feature

In [68]:
for file in files:
    with open(file) as f:
        gj = geojson.load(f)

    for feature in gj.get("features", []):
        feature = normalize_geometry(feature)

    # optional: overwrite or save new file
    with open(file.replace(".geojson", ".geojson"), "w") as f:
        geojson.dump(gj, f)

In [69]:
import geojson
import os

def normalize_geojson(input_path, output_path):
    with open(input_path) as f:
        gj = geojson.load(f)

    for feature in gj.get("features", []):
        props = feature.get("properties", {})

        # 1. Rename NAME -> name (if exists)
        if "NAME" in props:
            props["name"] = props.pop("NAME")

        # 2. Handle Germany counties: krs_name (list) -> name (string)
        if "krs_name" in props:
            krs_value = props["krs_name"]
            
            # If it's a list, extract first element
            if isinstance(krs_value, list) and len(krs_value) > 0:
                props["name"] = krs_value[0]
            else:
                props["name"] = krs_value

        feature["properties"] = props

    # Save normalized file
    with open(output_path, "w") as f:
        geojson.dump(gj, f)

    print(f"Processed: {input_path} -> {output_path}")


# Process all files
normalized_dir = "/home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/normalized"
os.makedirs(normalized_dir, exist_ok=True)

for file_path, label in files.items():
    # output_path = os.path.join(normalized_dir, os.path.basename(file_path))
    normalize_geojson(file_path, file_path)

Processed: /home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/usa/congressional.geojson -> /home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/usa/congressional.geojson
Processed: /home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/usa/outline.geojson -> /home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/usa/outline.geojson
Processed: /home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/usa/states.geojson -> /home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/usa/states.geojson
Processed: /home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/france/counties.geojson -> /home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/france/counties.geojson
Processed: /home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/france/states.geojson -> /home/kuenem/Documents/development/lectures/Master Thesis/data/geojson/france/states.geojso